# CSI Preprocess for Kaggle

Notebook này xử lý `data/raw/data4.csv` theo đúng hướng raw CSI I/Q:
- đọc dữ liệu I/Q + label
- tách `features` và `label`
- tính amplitude + phase
- encode phase thành `sin/cos`
- lọc nhiễu CSI
- tạo artifact để `train.py` trên Kaggle chỉ còn train model

Một số cell có phần `nếu có` để notebook vẫn chạy được ngay cả khi dữ liệu CSI không có cột datetime hay categorical.

## 1) Import các thư viện cần thiết

Cell này nạp các thư viện dùng cho đọc dữ liệu, xử lý số, lưu artifact và trực quan hóa.

## Pipeline tổng quan

`data4.csv` là dữ liệu raw I/Q đã gộp sẵn với label ở cột cuối. Notebook này sẽ đọc trực tiếp file đó, rồi đi theo chuỗi xử lý:

CSV raw → tách I/Q + label → amp/phase → sin/cos → lọc nhiễu → lưu clean.csv

In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import amp_phase
from src.preprocess import apply_hampel, butterworth_lowpass

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

RANDOM_STATE = 42
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

## 2) Định nghĩa đường dẫn và kiểm tra file dữ liệu

Cell này đặt đường dẫn làm việc, tạo thư mục artifact và kiểm tra file `data4.csv` đã tồn tại hay chưa.

In [ ]:
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "data" / "raw"
ARTIFACT_DIR = ROOT / "artifacts" / "preprocess"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DATA4_PATH = DATA_DIR / "data4.csv"
CLEAN_PATH = DATA_DIR / "clean.csv"

print(f"ROOT: {ROOT}")
print(f"DATA4 exists: {DATA4_PATH.exists()} | size={DATA4_PATH.stat().st_size if DATA4_PATH.exists() else 'n/a'}")
print(f"CLEAN exists: {CLEAN_PATH.exists()} | size={CLEAN_PATH.stat().st_size if CLEAN_PATH.exists() else 'n/a'}")

## 3) Đọc nhanh vài dòng đầu của dữ liệu

Cell này đọc thử vài dòng đầu của `data4.csv` để xem cấu trúc dữ liệu ban đầu.

In [ ]:
sample_data4 = pd.read_csv(DATA4_PATH, header=None, nrows=5)
print("Preview shape:", sample_data4.shape)
display(sample_data4.head())

## 4) Tách label và đọc raw I/Q

Cell này đọc `data4.csv` theo đúng format raw CSI: `RSSI, I/Q..., label`, rồi tách label trước khi tính amp/phase.

## 4b) Thống kê cấu trúc file CSV thô

Cell này đọc toàn bộ `data4.csv` bằng `csv.reader` (không parse float) để thống kê:
- Tổng số dòng, tổng số ô
- Phân phối số cột mỗi dòng — phát hiện dòng lỗi gây `inhomogeneous shape`
- Phân phối nhãn (cột cuối)
- Tỉ lệ ô rỗng / không parse được

In [ ]:
from collections import Counter

# ── Đọc raw để thống kê (không parse float, chỉ đếm) ────────────────────────
_path = DATA4_PATH

col_lengths      = []
label_counter    = Counter()
empty_cell_count = 0
unparseable_feat = 0
total_cells      = 0

with open(_path, "r", newline="") as _f:
    _reader = csv.reader(_f)
    for _r in _reader:
        if not _r:
            continue
        col_lengths.append(len(_r))
        total_cells      += len(_r)
        empty_cell_count += sum(1 for v in _r if v.strip() == "")
        for v in _r[:-1]:   # tất cả trừ cột cuối (label)
            try:
                float(v)
            except ValueError:
                unparseable_feat += 1
        try:
            label_counter[int(float(_r[-1]))] += 1
        except Exception:
            label_counter["?"] += 1

col_series = pd.Series(col_lengths, name="n_cols")

# ── In thống kê tổng quan ────────────────────────────────────────────────────
print("=" * 55)
print(f"  File          : {_path.name}")
print(f"  Tổng số dòng  : {len(col_lengths):,}")
print(f"  Tổng số ô     : {total_cells:,}")
print("=" * 55)

print("\n── Phân phối số cột mỗi dòng ──")
col_dist = col_series.value_counts().sort_index()
print(col_dist.to_string())
print(f"\n  Min cols : {col_series.min()}")
print(f"  Max cols : {col_series.max()}")
print(f"  Mode     : {col_series.mode()[0]}")

expected_cols = int(col_series.mode()[0])
bad_rows = int((col_series != expected_cols).sum())
print(f"\n  Dòng có số cột ≠ mode ({expected_cols}): {bad_rows:,}  "
      f"({bad_rows / len(col_lengths) * 100:.2f}%)")

print("\n── Phân phối nhãn (cột cuối) ──")
label_df = (
    pd.DataFrame.from_dict(label_counter, orient="index", columns=["count"])
    .sort_index()
)
label_df["pct"] = (label_df["count"] / label_df["count"].sum() * 100).round(2)
print(label_df.to_string())

print("\n── Chất lượng dữ liệu ──")
print(f"  Ô rỗng          : {empty_cell_count:,}  "
      f"({empty_cell_count / total_cells * 100:.3f}%)")
print(f"  Ô không parse   : {unparseable_feat:,}  "
      f"({unparseable_feat / total_cells * 100:.3f}%)")

# ── Biểu đồ ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram số cột mỗi dòng
x_labels = col_dist.index.astype(str).tolist()
axes[0].bar(x_labels, col_dist.values, color="steelblue")
axes[0].set_title("Phân phối số cột mỗi dòng")
axes[0].set_xlabel("Số cột")
axes[0].set_ylabel("Số dòng")
for x, y in zip(x_labels, col_dist.values):
    axes[0].text(x, y + max(col_dist.values) * 0.01,
                 f"{y:,}", ha="center", va="bottom", fontsize=9)

# Bar chart nhãn
lbl_keys = [str(k) for k in label_df.index]
lbl_vals = label_df["count"].values
axes[1].bar(lbl_keys, lbl_vals, color="tomato")
axes[1].set_title("Phân phối nhãn")
axes[1].set_xlabel("Label")
axes[1].set_ylabel("Số dòng")
for x, y in zip(lbl_keys, lbl_vals):
    axes[1].text(x, y + max(lbl_vals) * 0.01,
                 f"{y:,}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## 5) Tính amplitude + phase rồi lọc nhiễu

Cell này chuyển raw I/Q sang amplitude + phase, encode phase bằng `sin/cos`, rồi lọc nhiễu CSI.

In [ ]:
def _read_merged_csv_iq(path: str | Path) -> tuple[np.ndarray, np.ndarray]:
    rows: list[list[float]] = []
    labels: list[int] = []
    with open(path, "r", newline="") as f:
        reader = csv.reader(f)
        for r in reader:
            if not r:
                continue
            while len(r) > 1 and r[-1] == "":
                r.pop()
            while len(r) >= 2 and r[-2] == "":
                r.pop(-2)
            if len(r) < 3:
                continue
            *rss_iq, lab = r
            parsed_feat = []
            for v in rss_iq:
                try:
                    parsed_feat.append(float(v))
                except Exception:
                    parsed_feat.append(0.0)
            try:
                labels.append(int(float(lab)))
            except Exception:
                labels.append(-1)
            rows.append(parsed_feat)
    if not rows:
        raise ValueError(f"No rows found in {path}")
    X = np.array(rows, dtype=np.float32)
    y = np.array(labels, dtype=np.int64)
    return X, y


def _prepare_from_iq_rows(x_rows: np.ndarray) -> np.ndarray:
    if x_rows.shape[1] < 3:
        raise ValueError("Not enough columns to compute I/Q amp/phase.")
    csi = x_rows[:, 1:]
    if csi.shape[1] % 2 != 0:
        raise ValueError(f"I/Q columns must be even, got {csi.shape[1]}. Check data4.csv format.")
    amp, phase = amp_phase.compute_amplitude_phase(csi, interleaved=True)

    USE_AMP_LOG = False
    if USE_AMP_LOG:
        amp_used = np.log(amp + 1e-8)
    else:
        amp_used = amp

    amp_mu = np.mean(amp_used, axis=0, keepdims=True)
    amp_sigma = np.std(amp_used, axis=0, keepdims=True) + 1e-8
    amp_norm = (amp_used - amp_mu) / amp_sigma

    phase_cos = np.cos(phase)
    phase_sin = np.sin(phase)
    return np.concatenate((amp_norm, phase_cos, phase_sin), axis=1)


def _clean_raw_csi(csi: np.ndarray, use_hampel: bool, cutoff: float) -> np.ndarray:
    if use_hampel:
        print("Applying Hampel filter...")
        csi = apply_hampel(csi)
    print("Applying Butterworth lowpass filter...")
    csi = butterworth_lowpass(csi, cutoff=cutoff)
    return csi


X_raw, y_raw = _read_merged_csv_iq(DATA4_PATH)
print("Raw I/Q shape:", X_raw.shape)
print("Label counts:")
print(pd.Series(y_raw).value_counts().sort_index())
print("Raw I/Q preview:")
display(pd.DataFrame(X_raw[:5]).head())

In [ ]:
X_amp_phase = _prepare_from_iq_rows(X_raw)
print("Combined feature shape before filtering:", X_amp_phase.shape)

denoised = _clean_raw_csi(X_amp_phase, use_hampel=True, cutoff=0.1)
print("Feature shape after filtering:", denoised.shape)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, idx in zip(axes.flat, range(min(4, denoised.shape[1]))):
    ax.hist(X_amp_phase[:, idx], bins=40, alpha=0.6, label="before", color="steelblue")
    ax.hist(denoised[:, idx], bins=40, alpha=0.6, label="after", color="tomato")
    ax.set_title(f"Feature {idx}")
    ax.legend()
plt.tight_layout()
plt.show()

## 6) Lưu clean.csv để train.py xử lý window/split/train

Notebook dừng ở đây: phần sliding window, split train/val, augmentation và train nên để `train.py` trên Kaggle làm tiếp.

In [ ]:
clean_df = pd.DataFrame(denoised)
clean_df["label"] = y_raw.astype(np.int64)
clean_df.to_csv(CLEAN_PATH, index=False, header=False)

print(f"Saved clean dataset to {CLEAN_PATH}")
print("Clean shape:", clean_df.shape)
print("Label counts in clean.csv:")
print(clean_df["label"].value_counts().sort_index())
print("Clean preview:")
display(clean_df.head())
display(clean_df.tail())